# 深度学习教学案例：基于 GRU 的莎士比亚风格文本生成

## 1. 案例背景与知识点简介

### 什么是文本生成？
文本生成（Text Generation）是自然语言处理（NLP）中的核心任务之一。其目标是训练一个模型，使其能够根据已有的上下文预测下一个字符或单词，从而创作出具有逻辑性和风格一致性的文本。

### 为什么选择 GRU (门控循环单元)？
在处理序列数据（如文本、音频）时，传统的神经网络无法记忆之前的状态。**循环神经网络 (RNN)** 引入了循环结构来解决这一问题，但 RNN 在长序列中容易出现**梯度消失**（Gradient Vanishing），导致它“记不住”太久之前的信息。

**GRU (Gated Recurrent Unit)** 是对 RNN 的重要改进：
- **更简洁的门控结构**：相比 LSTM（长短期记忆网络），GRU 只有两个门：**更新门 (Update Gate)** 和 **重置门 (Reset Gate)**。
- **效率更优**：参数量更少，计算速度更快，且在许多中小规模数据集上的表现与 LSTM 相当。
- **机制原理**：更新门控制前一时刻状态流入当前状态的程度，重置门控制前一时刻状态对当前候选状态的影响程度。

### 设计程序的意义
1. **理解字符级建模**：学习如何将文字转化为机器可理解的张量（Tensor）。
2. **掌握序列处理逻辑**：实践“滑动窗口”采样和“自回归”生成的完整闭环。
3. **探索模型调优**：理解超参数（如隐藏层维度、层数）以及采样技巧（如温度系数）对生成效果的影响。

---

## 2. 环境准备与全局配置

在这一步，我们将导入 PyTorch 等核心库，并定义一个 `Config` 类。**统一管理超参数**是良好的工程实践，方便后续快速实验和参数调优。

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import os
import pickle
import sys
from tqdm import tqdm
import random

# --- 超参数与路径配置 ---
class Config:
    # 路径配置
    WORK_DIR = os.getcwd()
    DATA_DIR = os.path.join(WORK_DIR, "data")
    RAW_DATA_DIR = os.path.join(DATA_DIR, "raw")
    CSV_FILE = os.path.join(RAW_DATA_DIR, "Shakespeare_data.csv")
    VOCAB_FILE = os.path.join(WORK_DIR, "vocab.pkl")
    MODEL_DIR = os.path.join(WORK_DIR, "models")
    MODEL_SAVE_PATH = os.path.join(MODEL_DIR, "gru_model.pth")

    # 训练超参数
    BATCH_SIZE = 128
    SEQUENCE_LENGTH = 100  # 序列跨度
    N_EPOCHS = 10          # 演示建议 10-20
    LEARNING_RATE = 0.0005
    EMBED_DIM = 256
    HIDDEN_DIM = 512
    N_LAYERS = 2

def check_data_exists():
    """检查数据文件是否存在，仿照 data_loader.py 的逻辑"""
    if not os.path.exists(Config.CSV_FILE):
        print("=" * 50)
        print("错误：未找到数据文件！")
        print(f"配置文件路径: {Config.CSV_FILE}")
        print("请确保已下载并放置 Shakespeare_data.csv")
        print("=" * 50)
        return False
    return True

# 创建目录
os.makedirs(Config.MODEL_DIR, exist_ok=True)

print(f"当前设备: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if not check_data_exists():
    print("警告：后续单元格运行会因缺少数据报错。")

当前设备: cuda


## 3. 字符级词汇表构建与数据清洗

在自然语言处理中，原始文本是无法直接输入模型的。我们需要将其转化为数值。通常有两种方式：词级（Word-level）和字符级（Character-level）。

**字符级建模的特点：**
- **词汇表小**：仅包含字母、数字和标点符号（通常几十到几百个）。
- **处理灵活**：无需分词，且天然能处理“未登录词”（OOV）。
- **挑战**：序列长度较长，模型需要更强的捕捉长程依赖的能力。

### 本步骤的核心任务：
1. **清洗数据**：剔除 CSV 中的缺失值（NaN），这些通常是数据采集时的空行或格式错误。
2. **字符去重与排序**：获取文本中所有出现过的唯一字符，建立从“字符”到“索引”以及“索引”到“字符”的双向映射。
3. **持久化**：使用 `pickle` 保存词汇表。在神经网络部署时，如果没有原始的映射表，预测出的数字将无法还原为文字。

---

## 4. PyTorch Dataset 与 DataLoader 封装

为了高效进行深度学习训练，我们需要利用 **滑动窗口 (Sliding Window)** 技术构造样本。

### 数据构造逻辑：
假设我们的文本是 "SHAKESPEARE"，序列长度设定为 5：
- **输入序列 (X)**: `S H A K E`
- **目标序列 (y)**: `H A K E S` (即 X 中每个位置的下一个对应字符)

通过 PyTorch 的 `Dataset` 类，我们可以实现按需读取。而 `DataLoader` 则负责将这些样本打包成 **Batch**，并提供**洗牌 (Shuffle)** 功能，这对于训练的泛化能力至关重要。

In [9]:
def prepare_data_and_loaders(batch_size=128, sequence_length=100):
    # 1. 检查数据
    if not os.path.exists(Config.CSV_FILE):
        raise FileNotFoundError(f"未找到数据文件: {Config.CSV_FILE}")

    # 2. 加载数据
    df = pd.read_csv(Config.CSV_FILE)
    player_lines_df = df.dropna(subset=['Player'])
    text = player_lines_df['PlayerLine'].str.cat(sep='\n')
    print(f"数据加载完成，总字符数: {len(text)}")

    # 3. 创建字符集
    chars = sorted(list(set(text)))
    vocab_size = len(chars)
    char_to_ix = {ch: i for i, ch in enumerate(chars)}
    ix_to_char = {i: ch for i, ch in enumerate(chars)}

    # 保存词汇表
    with open(Config.VOCAB_FILE, 'wb') as f:
        pickle.dump((char_to_ix, ix_to_char, vocab_size), f)
    print(f"词汇表已保存，词汇量: {vocab_size}")

    # 4. 转换文本为整数
    text_as_int = [char_to_ix[ch] for ch in text]

    # 5. 滑动窗口创建序列 (仅演示前10万个字符以节省内存/时间)
    limit = 100000 
    input_sequences = []
    target_sequences = []
    for i in range(0, min(len(text_as_int) - sequence_length, limit)):
        input_sequences.append(text_as_int[i: i + sequence_length])
        target_sequences.append(text_as_int[i + 1: i + sequence_length + 1])

    # 6. 自定义 Dataset
    class ShakespeareDataset(Dataset):
        def __init__(self, sequences, targets):
            self.sequences = sequences
            self.targets = targets
        def __len__(self):
            return len(self.sequences)
        def __getitem__(self, idx):
            return torch.tensor(self.sequences[idx], dtype=torch.long), \
                   torch.tensor(self.targets[idx], dtype=torch.long)

    dataset = ShakespeareDataset(input_sequences, target_sequences)

    # 7. 划分数据集
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, vocab_size

# 执行预处理并输出中间结果
train_loader, val_loader, vocab_size = prepare_data_and_loaders(Config.BATCH_SIZE, Config.SEQUENCE_LENGTH)
print(f"训练 Batch 数: {len(train_loader)}")
sample_input, sample_target = next(iter(train_loader))
print(f"输入张量形状: {sample_input.shape}") # (Batch, Seq)

数据加载完成，总字符数: 4366022
词汇表已保存，词汇量: 77
训练 Batch 数: 625
输入张量形状: torch.Size([128, 100])


## 5. 构建 GRU 网络结构：深入理解层级逻辑

我们的深度学习模型由以下三部分组成：

1. **Embedding 层 (词嵌入)**：
   - 作用：将稀疏的整数索引转变为密集的高维连续向量。
   - 意义：在向量空间中，意思相近的字符（如 'a' 和 'A'）可以在训练过程中自发地聚合在一起。

2. **GRU 层 (核心大脑)**：
   - 作用：接收 Embedding 的输出，并结合上一时刻的“隐藏状态 (Hidden State)”来提取时序特征。
   - 重点：`batch_first=True` 意味着输入张量的形状为 `(Batch, Seq, Feature)`。

3. **Linear 层 (分类头)**：
   - 作用：将 GRU 输出的特征映射回词汇表的大小。
   - 输出：每个位置预测各字符出现的概率分布。

### 隐藏状态初始化的重要性：
RNN 类模型是有状态的。每当一个新序列开始训练时，我们需要一个初始的 $\mathbf{h}_0$。如果这个值在每个 batch 间被错误堆叠，会导致梯度流混乱。我们在模型中显式定义了 `init_hidden` 来重置状态。

---
以下是模型的代码定义：

In [10]:
class GRUModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, num_layers=2):
        super(GRUModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        # x: (batch, seq)
        embedded = self.embedding(x)  # (batch, seq, embed)
        out, hidden = self.gru(embedded, hidden) # out: (batch, seq, hidden)
        
        # 展平输出以便通过全连接层
        out = out.reshape(-1, self.hidden_dim) # (batch*seq, hidden)
        out = self.fc(out) # (batch*seq, vocab_size)
        return out, hidden

    def init_hidden(self, batch_size, device):
        return torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)

# 实例化模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GRUModel(
    vocab_size=vocab_size,
    embed_dim=Config.EMBED_DIM,
    hidden_dim=Config.HIDDEN_DIM,
    num_layers=Config.N_LAYERS
).to(device)

print(model)

GRUModel(
  (embedding): Embedding(77, 256)
  (gru): GRU(256, 512, num_layers=2, batch_first=True)
  (fc): Linear(in_features=512, out_features=77, bias=True)
)


## 6. 训练回路设计：技巧与细节

在模型训练中，除了前向传播和反向传播，有两项关键操作对循环神经网络至关重要：

1. **梯度裁剪 (Gradient Clipping)**：
   - 痛点：RNN 模型在展开时极易产生“梯度爆炸”，导致 Loss 变为 `NaN`。
   - 方案：通过 `torch.nn.utils.clip_grad_norm_` 将梯度的范数限制在一个阈值（如 5.0）内。

2. **状态重置**：
   - 每一个新 Batch 的数据在逻辑上可能与前一个 Batch 没有连续性，因此必须在循环开始前清空隐藏状态。

---

## 7. 文本生成推理：采样与 Temperature 系数

模型训练完成后，它不仅可以用索引输出下一个字符，还能通过**概率采样**来创作。

### 什么是 Temperature (温度)？
温度 $T$ 控制着分布的“锐度”：
- **$T = 1.0$**: 保持原始概率分布。
- **$T < 1.0$ (低温)**: 倾向于选择概率最高的字符。生成的文本非常稳健，但可能陷入死循环或显得死板。
- **$T > 1.0$ (高温)**: 平滑概率分布。模型会尝试一些低概率的字符组合，生成的文本更有“创意”，但也容易出现胡言乱语。

这一过程被称为 **自回归 (Autoregression)**：生成的上一个字符会作为下一轮的输入，不断迭代。

---
开始执行训练与生成演示：

In [11]:

def train_model(model, train_loader, epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=Config.LEARNING_RATE)
    
    print("开始训练模型...")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for inputs, targets in pbar:
            inputs, targets = inputs.to(device), targets.to(device)
            
            # --- 关键一：初始化隐藏状态 ---
            # 这是一个有状态模型，每个 batch 开始需要初始化
            hidden = model.init_hidden(inputs.size(0), device)
            
            optimizer.zero_grad()
            
            # 前向传播
            output, hidden = model(inputs, hidden)
            
            # 计算损失 (展平 targets)
            loss = criterion(output, targets.view(-1))
            
            # 反向传播
            loss.backward()
            
            # --- 关键二：梯度裁剪 ---
            # 位置必须在 backward() 之后，optimizer.step() 之前
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} 完成，平均损失: {avg_loss:.4f}")
        
        # 保存权重存档
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss
        }, Config.MODEL_SAVE_PATH)

# 执行训练
train_model(model, train_loader, epochs=Config.N_EPOCHS)


# --- 生成逻辑 ---
def generate_text(model, start_str="FROM ", gen_length=200, temperature=0.8, load_weights=False):
    model.eval()
    
    # 如果指定加载，则从文件载入权重
    if load_weights:
        if os.path.exists(Config.MODEL_SAVE_PATH):
            checkpoint = torch.load(Config.MODEL_SAVE_PATH, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            print("已加载保存的模型权重。")
    
    with open(Config.VOCAB_FILE, 'rb') as f:
        char_to_ix, ix_to_char, _ = pickle.load(f)
    
    input_seq = [char_to_ix[ch] for ch in start_str]
    input_tensor = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0).to(device)
    hidden = model.init_hidden(1, device)
    
    generated_text = start_str
    
    with torch.no_grad():
        for _ in range(gen_length):
            output, hidden = model(input_tensor, hidden)
            
            # 提取最后一位预测
            last_output = output[-1] 
            
            # 应用 Temperature 采样
            probs = torch.softmax(last_output / temperature, dim=0)
            top_i = torch.multinomial(probs, 1)[0]
            
            char = ix_to_char[top_i.item()]
            generated_text += char
            
            # 自回归输入
            input_tensor = torch.tensor([[top_i]], dtype=torch.long).to(device)
            
    return generated_text

# 展示成果
print("\n" + "="*30)
print("生成样例文本 (Temperature=0.7):")
print("-" * 30)
print(generate_text(model, start_str="Shall I compare thee ", temperature=0.7))
print("="*30)

开始训练模型...


Epoch 1/10: 100%|██████████| 625/625 [00:32<00:00, 19.15it/s, loss=0.7772]


Epoch 1 完成，平均损失: 1.5221


Epoch 2/10: 100%|██████████| 625/625 [00:32<00:00, 19.00it/s, loss=0.1796]


Epoch 2 完成，平均损失: 0.3505


...

Epoch 8 完成，平均损失: 0.1131


Epoch 9/10: 100%|██████████| 625/625 [00:32<00:00, 19.03it/s, loss=0.1104]


Epoch 9 完成，平均损失: 0.1111


Epoch 10/10: 100%|██████████| 625/625 [00:32<00:00, 19.10it/s, loss=0.1103]


Epoch 10 完成，平均损失: 0.1095

生成样例文本 (Temperature=0.7):
------------------------------
Shall I compare thee as I fear the
roaring of a lion's whelp.
And why not as the lion?
The king is to be feared as the lion: dost thou
think I'll fear thee as I fear thy father? nay, an
I do, I pray God my girdle break.
O


In [ ]:
#我是否应该把你比作我所害怕的小狮子的咆哮。为何不把你比作狮子呢？
#国王应当像狮子一样令人畏惧：你以为我会像害怕你父亲那样害怕你吗？
#不，即使我害怕，也祈求上帝让我腰带断裂。

## 8. 教学拓展与深度思考

本案例展示了基于字符的文本生成闭环。为了让学生更深入地掌握，可以思考以下进阶方向：

### 思考题集：

**1. 架构选择：为什么在本案中使用 GRU 而不是 LSTM 或普通的 RNN？**
- *解答提示*：普通的 RNN 容易梯度消失；LSTM 参数过多且计算复杂；GRU 在保持长程捕捉能力的同时更轻量化，非常适合这种规模的文本生成实验。

**2. 损失函数：交叉熵损失 (CrossEntropyLoss) 的输入到底是什么？**
- *解答提示*：代码中我们将输出 `output` 变成了 `(Batch * Seq, VocabSize)`，将目标 `targets` 变成了 `(Batch * Seq)`。这实际上是在把序列里的每一个字符预测都看作一个独立的分类任务。

**3. 生成优化：束搜索 (Beam Search) 与多项式采样 (Multinomial Sampling) 有何区别？**
- *解答提示*：本代码使用的是多项式采样（带温度调节），具有很好的随机性和多样性。而 Beam Search 则旨在找到一条整体概率最大的路径，生成的句子通常更通顺但缺乏灵性。

### 实验建议：
- 尝试修改 `Config.SEQUENCE_LENGTH`（如改为 50 或 200），观察 Loss 的下降速度。
- 将 `Config.TEMPERATURE` 分别设为 0.1, 0.8, 2.0，对比生成出的“诗句”风格差异。